# Phase 1 · Notebook 02 — Exploratory Data Analysis (EDA)
**Project:** Water Quality Monitoring Analytics · **Dataset:** Water Quality Portal (WQP), 2024–2025
**Input:** `wq_curated.parquet` produced by `01_data_profiling_and_curation.ipynb` (only rows with `IsAnalysisReady = True` are analysed)

This notebook covers **Part A (second half): Exploratory Data Analysis** (assignment §5) and gives first answers to the **mandatory business questions** (§3) that EDA can address:

| Section | EDA area (§5) | Business questions (§3) |
|---|---|---|
| 2 | A. Numerical analysis | — |
| 3 | B. Categorical analysis | Q8 location / media types |
| 4 | C. Temporal analysis | Q1 over time · Q4 seasonal |
| 5 | D. Geographic / entity analysis | Q2 compare sites · Q3 extreme sites · Q7 coverage |
| 6 | E. Outcome analysis (volume vs rate) | Q6 exceedances |
| 7 | Cross-parameter relationships | Q5 pH, DO, turbidity, nutrients |
| 8 | Summary of findings | Q9, Q10 → Phase 3 |

> All values are in canonical units: °C, mg/L (DO), pH units, NTU, µS/cm, mg/L as N.
> Statements describe **associations** in observational monitoring data. They are **not** causal claims.

## 1. Setup and load the curated data

In [ ]:
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

IN_KAGGLE = Path("/kaggle/input").exists()
if IN_KAGGLE:
    PROJECT_ROOT = Path("/kaggle/working")
else:
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / "notebooks").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent
TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
for d in (TABLE_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)


def find_curated():
    # Full curated file first; the sample file is only used if the full one does not exist.
    patterns = [str(PROJECT_ROOT / "data" / "processed" / "wq_curated.parquet"),
                "/kaggle/input/**/wq_curated.parquet",
                str(PROJECT_ROOT / "data" / "processed" / "wq_curated_sample.parquet"),
                "/kaggle/input/**/wq_curated_sample.parquet"]
    for p in patterns:
        hits = glob.glob(p, recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError("Run 01_data_profiling_and_curation.ipynb first (or attach its output on Kaggle).")


CURATED = find_curated()
cur = pd.read_parquet(CURATED)
df = cur[cur["IsAnalysisReady"]].copy()
for c in df.select_dtypes("category"):
    df[c] = df[c].cat.remove_unused_categories()
print(f"Curated file : {CURATED}")
print(f"Curated rows : {len(cur):,} | analysis-ready rows used here: {len(df):,} ({100 * len(df) / len(cur):.1f}%)")

In [ ]:
# ---- shared style: one fixed colour per parameter (same as notebook 01) ----
PARAM_ORDER = ["Temperature", "DO", "pH", "Turbidity", "Conductivity", "Nitrate"]
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
PARAM_COLORS = dict(zip(PARAM_ORDER, PALETTE))
UNITS = {"Temperature": "°C", "DO": "mg/L", "pH": "pH units", "Turbidity": "NTU",
         "Conductivity": "µS/cm", "Nitrate": "mg/L as N"}
LOG_SCALE = {"Turbidity", "Conductivity", "Nitrate"}     # strongly right-skewed -> log axis for display
PRIMARY, CRITICAL, MUTED = PALETTE[0], "#d03b3b", "#8a8984"
SEASONS = ["Winter", "Spring", "Summer", "Fall"]

sns.set_theme(style="whitegrid", rc={
    "axes.edgecolor": "#c3c2b7", "grid.color": "#e6e5e0", "axes.titleweight": "bold",
    "axes.titlesize": 11, "axes.labelcolor": "#52514e", "xtick.color": "#52514e",
    "ytick.color": "#52514e", "figure.dpi": 100})


def save_fig(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=130, bbox_inches="tight")


def save_table(t, name):
    t.to_csv(TABLE_DIR / f"{name}.csv")


def param_grid(nrows=2, ncols=3, size=(15, 7.5), **kw):
    # Small multiples: one panel per parameter, because the parameters have different units.
    fig, axes = plt.subplots(nrows, ncols, figsize=size, **kw)
    return fig, dict(zip(PARAM_ORDER, axes.ravel()))


def plot_sample(frame, n=200_000, seed=42):
    # Random sample for scatter/histogram drawing only. Statistics always use all rows.
    return frame.sample(n, random_state=seed) if len(frame) > n else frame


df["P"] = pd.Categorical(df["ParameterShort"].astype(str), categories=PARAM_ORDER, ordered=True)
df["Season"] = pd.Categorical(df["Season"].astype(str), categories=SEASONS, ordered=True)
df["Exc"] = df["ExceedanceFlag"].astype("float64")     # float copies of the 0/1 flags for rates
df["HR"] = df["HighRiskFlag"].astype("float64")
print(df[["P", "ValueStd", "UnitStd", "StartDate", "SiteId", "RegionGroup", "ExceedanceFlag"]].head())

### 1.1 Dataset at a glance

In [ ]:
overview = pd.Series({
    "Analysis-ready results": f"{len(df):,}",
    "Period": f"{df['StartDate'].min().date()} to {df['StartDate'].max().date()}",
    "Monitoring sites": f"{df['SiteId'].nunique():,}",
    "Organizations": f"{df['OrganizationId'].nunique():,}",
    "Parameters": df["P"].nunique(),
    "Regions (derived)": df["RegionGroup"].nunique(),
    "Results with known time of day": f"{100 * df['TimeKnown'].mean():.1f}%",
    "Exceedances": f"{int(df['ExceedanceFlag'].sum()):,} ({100 * df['ExceedanceFlag'].mean():.2f}%)",
    "High-risk results": f"{int(df['HighRiskFlag'].sum()):,} ({100 * df['HighRiskFlag'].mean():.2f}%)",
}, name="Value")
save_table(overview.to_frame(), "02_overview")
overview.to_frame()

## 2. A — Numerical analysis
Per parameter: mean, median, min, max, variance, std, quartiles, IQR, skewness, kurtosis and coefficient of variation.
*Skewness > 1* means a long right tail (a few very high values). *Kurtosis > 3* (excess) means heavier tails than a normal distribution.

In [ ]:
g = df.groupby("P", observed=True)["ValueStd"]
num = pd.DataFrame({
    "Unit": pd.Series(UNITS),
    "Count": g.size(), "Mean": g.mean(), "Median": g.median(), "Min": g.min(), "Max": g.max(),
    "Variance": g.var(), "StdDev": g.std(), "Q1": g.quantile(0.25), "Q3": g.quantile(0.75),
    "Skewness": g.skew(), "Kurtosis(excess)": g.apply(lambda s: s.kurt()),
}).reindex(PARAM_ORDER)
num["IQR"] = num["Q3"] - num["Q1"]
num["CV(%)"] = 100 * num["StdDev"] / num["Mean"]
num = num[["Unit", "Count", "Mean", "Median", "Min", "Q1", "Q3", "IQR", "Max", "Variance", "StdDev",
           "CV(%)", "Skewness", "Kurtosis(excess)"]]
save_table(num, "02_A_numeric_summary")
num

In [ ]:
# Distribution of every parameter (histogram + KDE). Skewed parameters are drawn on a log axis.
fig, ax_of = param_grid()
for p, ax in ax_of.items():
    v = plot_sample(df.loc[df["P"] == p, "ValueStd"])
    log = p in LOG_SCALE
    if log:
        v = v[v > 0]
    sns.histplot(v, ax=ax, bins=60, color=PARAM_COLORS[p], log_scale=log, kde=True,
                 edgecolor="white", linewidth=0.3, line_kws={"linewidth": 2})
    med = df.loc[df["P"] == p, "ValueStd"].median()
    ax.axvline(med, color="#0b0b0b", linestyle="--", linewidth=1)
    ax.set_title(f"{p} (median {med:,.2f} {UNITS[p]})")
    ax.set_xlabel(UNITS[p] + (" (log scale, values > 0)" if log else ""))
    ax.set_ylabel("results")
fig.suptitle("A1. Distribution of each parameter", fontweight="bold", y=1.01)
fig.tight_layout()
save_fig("02_A1_distributions")
plt.show()

In [ ]:
# Box plots: one panel per parameter (different units). Whiskers = 1.5·IQR, outliers hidden for readability.
fig, ax_of = param_grid(size=(15, 6))
for p, ax in ax_of.items():
    v = df.loc[df["P"] == p, "ValueStd"]
    ax.boxplot(v.dropna(), vert=True, showfliers=False, widths=0.5, patch_artist=True,
               boxprops=dict(facecolor=PARAM_COLORS[p], alpha=0.85, edgecolor="#52514e"),
               medianprops=dict(color="#0b0b0b", linewidth=2), whiskerprops=dict(color="#52514e"),
               capprops=dict(color="#52514e"))
    n_out = ((v < v.quantile(.25) - 1.5 * (v.quantile(.75) - v.quantile(.25))) |
             (v > v.quantile(.75) + 1.5 * (v.quantile(.75) - v.quantile(.25)))).mean() * 100
    ax.set_title(f"{p}  ({n_out:.1f}% beyond whiskers)")
    ax.set_ylabel(UNITS[p])
    ax.set_xticks([])
fig.suptitle("A2. Box plots per parameter (fliers hidden)", fontweight="bold", y=1.02)
fig.tight_layout()
save_fig("02_A2_boxplots")
plt.show()

**How to read A:** temperature, DO and pH are roughly bell-shaped, so the mean ≈ the median. Turbidity, conductivity and nitrate are **strongly right-skewed** (high skewness and kurtosis): most results are low, with a few very high events. For those three the **median** and the log scale are the right summaries, and models in Phase 3 should use log-transformed values.

## 3. B — Categorical analysis
Frequency, percentage contribution, cumulative share, **rare categories** (< 1% of results), and cross-tabulations with the outcome (`ExceedanceFlag`).

In [ ]:
def freq_table(col, top=None):
    vc = df[col].value_counts()
    t = pd.DataFrame({"Results": vc, "Pct": 100 * vc / vc.sum()})
    t["CumPct"] = t["Pct"].cumsum()
    return t.head(top) if top else t


for col in ["P", "ParameterGroup", "SampleTypeGroup", "MediaSubdivision", "RegionGroup",
            "ResultStatus", "ValueType", "Provider", "TimeBand"]:
    t = freq_table(col)
    save_table(t, f"02_B_freq_{col}")
    print(f"\n=== {col} ===")
    display(t.style.format({"Results": "{:,}", "Pct": "{:.2f}", "CumPct": "{:.2f}"}))

In [ ]:
# Rare-category analysis: how many categories each hold < 1% of results, and their combined share
rare_rows = []
for col in ["OrganizationId", "ActivityType", "MediaSubdivision", "CollectionMethod", "SiteId", "SampleFraction"]:
    share = df[col].value_counts(normalize=True) * 100
    rare = share[share < 1]
    rare_rows.append({"Attribute": col, "Categories": len(share), "RareCategories(<1%)": len(rare),
                      "RareCombinedPct": rare.sum(), "Top category": share.index[0], "TopPct": share.iloc[0]})
rare_table = pd.DataFrame(rare_rows).set_index("Attribute")
save_table(rare_table, "02_B_rare_categories")
rare_table

In [ ]:
# Top 15 organizations by results (single series -> one colour, title names it)
org = freq_table("OrganizationId", top=15).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(org.index.astype(str), org["Results"], color=PRIMARY, height=0.65)
for y, (n, pct) in enumerate(zip(org["Results"], org["Pct"])):
    ax.text(n, y, f"  {pct:.1f}%", va="center", fontsize=8, color="#52514e")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x / 1e3:,.0f}k"))
ax.set_title(f"B1. Top 15 organizations by results ({org['Pct'].sum():.0f}% of all results)")
ax.set_xlabel("analysis-ready results")
ax.grid(axis="y", visible=False)
save_fig("02_B1_top_organizations")
plt.show()

In [ ]:
# Result-band distribution per parameter (ordered bands; small multiples because bands differ by parameter)
fig, ax_of = param_grid(size=(16, 7.5))
for p, ax in ax_of.items():
    s = df.loc[df["P"] == p, "ResultBand"].astype(str).value_counts(normalize=True) * 100
    order = [b for b in df.loc[df["P"] == p, "ResultBand"].astype(str).unique()]
    order = sorted(order, key=lambda b: df.loc[(df["P"] == p) & (df["ResultBand"] == b), "ValueStd"].median())
    s = s.reindex(order)
    ax.barh(s.index, s.values, color=PARAM_COLORS[p], height=0.6)
    for y, v in enumerate(s.values):
        ax.text(v, y, f" {v:.1f}%", va="center", fontsize=8, color="#52514e")
    ax.set_xlim(0, max(s.values) * 1.25)
    ax.set_title(p)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
fig.suptitle("B2. Share of results in each ResultBand (low → high)", fontweight="bold", y=1.01)
fig.tight_layout()
save_fig("02_B2_result_bands")
plt.show()

In [ ]:
# Cross-tabulations with the outcome: exceedance rate (%) by parameter x categorical attribute
def rate_crosstab(col):
    t = pd.crosstab(df[col], df["P"], values=df["ExceedanceFlag"].astype(float), aggfunc="mean") * 100
    n = pd.crosstab(df[col], df["P"])
    return t.where(n >= 100).round(2)          # rates from < 100 results are hidden (unstable)


for col in ["SampleTypeGroup", "MediaSubdivision", "RegionGroup"]:
    t = rate_crosstab(col)
    save_table(t, f"02_B_exceedance_rate_by_{col}")
    print(f"\n=== Exceedance rate (%) by {col} x parameter (blank = fewer than 100 results) ===")
    display(t)

**Q8 — Comparing monitoring-location types.** The result file has **no `MonitoringLocationTypeName`** (it is in the WQP *Station* file, joined in Phase 2). As an interim proxy we compare **media subdivision** (Surface Water / Groundwater / Ocean Water / …) and **sample type** (field measurement / logger / lab sample). The crosstabs above show that exceedance rates depend on how and where a result was measured, so the Phase 2 comparison by true site type needs the station enrichment.

In [ ]:
# Q8 proxy: medians by media subdivision (only groups with >= 100 results)
med = pd.crosstab(df["MediaSubdivision"], df["P"], values=df["ValueStd"], aggfunc="median")
cnt = pd.crosstab(df["MediaSubdivision"], df["P"])
q8 = med.where(cnt >= 100).round(2)
save_table(q8, "02_Q8_median_by_media")
q8

## 4. C — Temporal analysis (Q1 measurements over time, Q4 seasonal variation)
Order: yearly → quarterly → monthly → weekly → day-of-week → hour/time-band → peak/low periods → seasonal.
*Monitoring volume* (how much was sampled) and *water-quality level* (what was measured) are shown separately.

In [ ]:
yearly = df.groupby("Year").agg(Results=("ValueStd", "size"), Sites=("SiteId", "nunique"),
                                Orgs=("OrganizationId", "nunique"), MonthsCovered=("Month", "nunique"))
quarterly = df.groupby(["Year", "Quarter"]).agg(Results=("ValueStd", "size"), Sites=("SiteId", "nunique"))
save_table(yearly, "02_C_yearly")
save_table(quarterly, "02_C_quarterly")
display(yearly)
print("Note: compare years only for months covered in both. A partial year is not a decline.")
display(quarterly)

In [ ]:
monthly = (df.groupby("YearMonth", observed=True)
             .agg(Results=("ValueStd", "size"), Sites=("SiteId", "nunique"))
             .sort_index())
monthly.index = pd.PeriodIndex(monthly.index.astype(str), freq="M").to_timestamp()
save_table(monthly, "02_C_monthly_volume")

# two separate charts (different scales), never a dual axis
fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 4))
a1.plot(monthly.index, monthly["Results"], color=PRIMARY, linewidth=2, marker="o", markersize=4)
a1.set_title("C1. Results per month")
a1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x / 1e3:,.0f}k"))
a2.plot(monthly.index, monthly["Sites"], color=PRIMARY, linewidth=2, marker="o", markersize=4)
a2.set_title("C2. Active monitoring sites per month")
for a in (a1, a2):
    a.set_ylim(bottom=0)
    a.tick_params(axis="x", rotation=45)
fig.tight_layout()
save_fig("02_C1_monthly_volume")
plt.show()

peak, low = monthly["Results"].nlargest(3), monthly["Results"].nsmallest(3)
print("Peak sampling months :", ", ".join(f"{d:%b %Y} ({v:,})" for d, v in peak.items()))
print("Lowest sampling months:", ", ".join(f"{d:%b %Y} ({v:,})" for d, v in low.items()))

In [ ]:
# Monthly median of each parameter, with the inter-quartile band (Q1: change over time)
mstat = (df.groupby(["P", "YearMonth"], observed=True)["ValueStd"]
           .describe(percentiles=[.25, .5, .75])[["count", "25%", "50%", "75%"]].reset_index())
mstat["Month"] = pd.PeriodIndex(mstat["YearMonth"].astype(str), freq="M").to_timestamp()
save_table(mstat.set_index(["P", "YearMonth"]), "02_C_monthly_parameter_stats")

fig, ax_of = param_grid(size=(16, 8), sharex=True)
for p, ax in ax_of.items():
    s = mstat[(mstat["P"] == p) & (mstat["count"] >= 30)]
    ax.fill_between(s["Month"], s["25%"], s["75%"], color=PARAM_COLORS[p], alpha=0.18, linewidth=0)
    ax.plot(s["Month"], s["50%"], color=PARAM_COLORS[p], linewidth=2, marker="o", markersize=3)
    ax.set_title(f"{p} — monthly median (band = IQR)")
    ax.set_ylabel(UNITS[p])
    ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
save_fig("02_C3_monthly_parameter_trend")
plt.show()

In [ ]:
# Weekly volume and day-of-week pattern
weekly = df.groupby("WeekStart").size()
dow = df["DayOfWeek"].value_counts().reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday",
                                               "Saturday", "Sunday"])
dow_by_type = pd.crosstab(df["DayOfWeek"], df["SampleTypeGroup"], normalize="columns") * 100
save_table(weekly.to_frame("Results"), "02_C_weekly_volume")
save_table(dow_by_type, "02_C_day_of_week_share_by_sampletype")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 4), gridspec_kw={"width_ratios": [2, 1]})
a1.plot(weekly.index, weekly.values, color=PRIMARY, linewidth=1.5)
a1.set_title("C4. Results per week")
a1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x / 1e3:,.0f}k"))
a1.set_ylim(bottom=0)
a2.bar(dow.index.str[:3], dow.values, color=[PRIMARY] * 5 + [MUTED] * 2, width=0.65)
a2.set_title("C5. Results by day of week (weekend in grey)")
a2.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x / 1e3:,.0f}k"))
a2.grid(axis="x", visible=False)
fig.tight_layout()
save_fig("02_C4_weekly_dow")
plt.show()
display(dow_by_type.round(1))

In [ ]:
# Hour-of-day pattern (local time, rows with a known time only)
timed = df[df["TimeKnown"]]
hourly = timed.groupby(["Hour", "SampleTypeGroup"], observed=True).size().unstack(fill_value=0)
save_table(hourly, "02_C_hourly_by_sampletype")
tb = df["TimeBand"].value_counts(normalize=True).mul(100).round(2)
display(tb.to_frame("Pct of results"))

# discrete visits (field + lab) vs loggers: different sampling regimes, so separate panels
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for ax, grp in zip(axes, [["Field Measurement", "Lab Sample"], ["Logger / Continuous"]]):
    s = hourly[[c for c in grp if c in hourly]].sum(axis=1).reindex(range(24), fill_value=0)
    ax.bar(s.index, s.values, color=PRIMARY, width=0.75)
    ax.set_title("C6. Hour of day: " + " + ".join(grp))
    ax.set_xlabel("hour (local time)")
    ax.set_xticks(range(0, 24, 2))
    ax.grid(axis="x", visible=False)
fig.tight_layout()
save_fig("02_C6_hourly")
plt.show()

In [ ]:
# Diurnal cycle of temperature and DO (median by hour). A physical check: DO usually peaks in the afternoon
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, p in zip(axes, ["Temperature", "DO", "pH"]):
    s = timed[timed["P"] == p].groupby("Hour")["ValueStd"].agg(["median", "size"])
    s = s[s["size"] >= 200]
    ax.plot(s.index, s["median"], color=PARAM_COLORS[p], linewidth=2, marker="o", markersize=4)
    ax.set_title(f"C7. {p}: median by hour")
    ax.set_xlabel("hour (local)")
    ax.set_ylabel(UNITS[p])
    ax.set_xticks(range(0, 24, 3))
fig.tight_layout()
save_fig("02_C7_diurnal")
plt.show()

### 4.1 Seasonal variation (Q4)

In [ ]:
seasonal = df.pivot_table(index="P", columns="Season", values="ValueStd", aggfunc="median", observed=True)
seasonal_n = df.pivot_table(index="P", columns="Season", values="ValueStd", aggfunc="size", observed=True)
seasonal["Summer−Winter"] = seasonal["Summer"] - seasonal["Winter"]
save_table(seasonal, "02_C_seasonal_median")
display(seasonal.round(2))

fig, ax_of = param_grid(size=(16, 8))
for p, ax in ax_of.items():
    s = plot_sample(df[df["P"] == p], 150_000)
    sns.boxplot(data=s, x="Season", y="ValueStd", order=SEASONS, ax=ax, showfliers=False,
                color=PARAM_COLORS[p], width=0.55, linewidth=1)
    if p in LOG_SCALE:
        ax.set_yscale("symlog", linthresh=1)
    ax.set_title(f"{p} by season")
    ax.set_ylabel(UNITS[p])
    ax.set_xlabel("")
fig.suptitle("C8. Seasonal distribution per parameter (fliers hidden)", fontweight="bold", y=1.01)
fig.tight_layout()
save_fig("02_C8_seasonal_boxplots")
plt.show()

In [ ]:
# Exceedance rate by season x parameter (heat map, sequential single-hue scale)
season_rate = 100 * df.pivot_table(index="P", columns="Season", values="Exc", aggfunc="mean", observed=True)
save_table(season_rate, "02_C_seasonal_exceedance_rate")
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(season_rate, annot=True, fmt=".1f", cmap="Blues", cbar_kws={"label": "exceedance rate (%)"},
            linewidths=2, linecolor="white", ax=ax)
ax.set_title("C9. Exceedance rate (%) by season")
ax.set_xlabel("")
ax.set_ylabel("")
save_fig("02_C9_seasonal_exceedance_heatmap")
plt.show()

## 5. D — Geographic and entity analysis (Q2, Q3, Q7)
Entities: **monitoring sites**, **regions** (derived from coordinates), **organizations**.
Site coordinates = median of the valid activity coordinates. Sites without valid coordinates are left off the map.

In [ ]:
site = (df.groupby("SiteId", observed=True)
          .agg(Lat=("Latitude", "median"), Lon=("Longitude", "median"), Region=("RegionGroup", "first"),
               Org=("OrganizationId", "first"), Results=("ValueStd", "size"),
               Params=("P", "nunique"), Days=("StartDate", "nunique"),
               ExceedRate=("ExceedanceFlag", lambda s: 100 * s.astype(float).mean()),
               HighRiskRate=("HighRiskFlag", lambda s: 100 * s.astype(float).mean())))
REGION_ORDER = ["West (Pacific)", "Mountain / Plains", "Central", "East", "Alaska",
                "Hawaii / Pacific Islands", "Caribbean", "Unknown"]
REGION_COLORS = dict(zip(REGION_ORDER, PALETTE[:7] + [MUTED]))
print(f"{len(site):,} sites; {site['Lat'].notna().sum():,} with valid coordinates")

fig, ax = plt.subplots(figsize=(14, 7))
for r in REGION_ORDER:
    s = site[(site["Region"] == r) & site["Lat"].notna()]
    if len(s):
        ax.scatter(s["Lon"], s["Lat"], s=4, color=REGION_COLORS[r], label=f"{r} ({len(s):,})", alpha=0.7, linewidths=0)
ax.set_xlim(-170, -60)
ax.set_ylim(15, 72)
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title("D1. Monitoring sites (one dot per site), coloured by derived RegionGroup")
ax.legend(markerscale=4, frameon=False, loc="lower left", fontsize=9)
save_fig("02_D1_site_map")
plt.show()

In [ ]:
region = (df.groupby("RegionGroup", observed=True)
            .agg(Results=("ValueStd", "size"), Sites=("SiteId", "nunique"), Orgs=("OrganizationId", "nunique"),
                 Exceedances=("ExceedanceFlag", "sum"), HighRisk=("HighRiskFlag", "sum")))
region["ResultsPerSite"] = region["Results"] / region["Sites"]
region["ExceedRatePct"] = 100 * region["Exceedances"] / region["Results"]
region["HighRiskRatePct"] = 100 * region["HighRisk"] / region["Results"]
region = region.reindex([r for r in REGION_ORDER if r in region.index])
save_table(region, "02_D_region_summary")
region

In [ ]:
# Q2 — compare parameters across sites: distribution of SITE MEDIANS (sites with >= 30 results of that parameter)
MIN_N = 30
sp = (df.groupby(["SiteId", "P"], observed=True)["ValueStd"].agg(["median", "size"]).reset_index())
sp = sp[sp["size"] >= MIN_N]
site_spread = sp.groupby("P", observed=True)["median"].describe(percentiles=[.05, .25, .5, .75, .95])
site_spread.insert(0, "Unit", pd.Series(UNITS))
save_table(site_spread, "02_Q2_site_median_spread")
print(f"Spread of site medians across sites with >= {MIN_N} results per parameter:")
display(site_spread)

fig, ax_of = param_grid(size=(16, 7.5))
for p, ax in ax_of.items():
    v = sp.loc[sp["P"] == p, "median"]
    log = p in LOG_SCALE
    sns.histplot(v[v > 0] if log else v, bins=50, ax=ax, color=PARAM_COLORS[p], log_scale=log,
                 edgecolor="white", linewidth=0.3)
    ax.set_title(f"{p}: site medians ({len(v):,} sites)")
    ax.set_xlabel(UNITS[p])
    ax.set_ylabel("sites")
fig.suptitle("D2. How much do sites differ? (distribution of per-site medians)", fontweight="bold", y=1.01)
fig.tight_layout()
save_fig("02_D2_site_medians")
plt.show()

In [ ]:
# Q3 — sites with extreme / poor-quality measurements
# Poor-quality site: exceedance rate >= 25% for a parameter with >= 30 results (documented rule)
POOR_RATE = 25
sx = (df.groupby(["SiteId", "P"], observed=True)
        .agg(Results=("ValueStd", "size"), Median=("ValueStd", "median"),
             Exceed=("ExceedanceFlag", "sum"), HighRisk=("HighRiskFlag", "sum")).reset_index())
sx = sx[sx["Results"] >= MIN_N]
sx["ExceedRatePct"] = 100 * sx["Exceed"] / sx["Results"]
sx["HighRiskRatePct"] = 100 * sx["HighRisk"] / sx["Results"]
poor = sx[sx["ExceedRatePct"] >= POOR_RATE]
poor_summary = poor.groupby("P", observed=True).agg(PoorSites=("SiteId", "nunique")).join(
    sx.groupby("P", observed=True).agg(SitesEvaluated=("SiteId", "nunique")))
poor_summary["PoorPct"] = 100 * poor_summary["PoorSites"] / poor_summary["SitesEvaluated"]
save_table(poor_summary, "02_Q3_poor_quality_sites_by_parameter")
print(f"Poor-quality sites (>= {POOR_RATE}% exceedances, >= {MIN_N} results), by parameter:")
display(poor_summary)

names = cur.drop_duplicates("SiteId").set_index("SiteId")[["SiteName", "OrganizationId", "RegionGroup"]]
worst = (sx.sort_values(["HighRiskRatePct", "Results"], ascending=False)
           .groupby("P", observed=True).head(5).join(names, on="SiteId"))
worst = worst[["P", "SiteId", "SiteName", "OrganizationId", "RegionGroup", "Results", "Median",
               "ExceedRatePct", "HighRiskRatePct"]].sort_values(["P", "HighRiskRatePct"], ascending=[True, False])
save_table(worst.set_index(["P", "SiteId"]), "02_Q3_worst_sites")
print("Top 5 sites per parameter by high-risk rate:")
worst

In [ ]:
# Q7 — sampling coverage by organization and by site
org_cov = (df.groupby("OrganizationId", observed=True)
             .agg(Results=("ValueStd", "size"), Sites=("SiteId", "nunique"), Params=("P", "nunique"),
                  Months=("YearMonth", "nunique"), Regions=("RegionGroup", "nunique"))
             .sort_values("Results", ascending=False))
org_cov["ResultsPerSite"] = org_cov["Results"] / org_cov["Sites"]
save_table(org_cov, "02_Q7_organization_coverage")
print(f"{len(org_cov)} organizations. Top 10 contribute {100 * org_cov['Results'].head(10).sum() / len(df):.1f}% of results")
display(org_cov.head(15))

cov = site["Params"].value_counts().sort_index()
days = site["Days"].describe(percentiles=[.25, .5, .75, .9])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4))
a1.bar(cov.index.astype(str), cov.values, color=PRIMARY, width=0.6)
a1.set_title("D3. Sites by number of parameters measured (of 6)")
a1.set_xlabel("parameters measured at the site")
a1.set_ylabel("sites")
a1.grid(axis="x", visible=False)
d = site["Days"].clip(upper=site["Days"].quantile(0.99))
a2.hist(d, bins=50, color=PRIMARY, edgecolor="white", linewidth=0.3)
a2.set_title(f"D4. Sampling days per site (median {days['50%']:.0f})")
a2.set_xlabel("distinct sampling days (2024–2025, clipped at p99)")
a2.set_ylabel("sites")
fig.tight_layout()
save_fig("02_D3_site_coverage")
plt.show()

In [ ]:
# Site x Time heat map: monthly results for the 30 most-sampled sites (coverage gaps become visible)
top_sites = site.nlargest(30, "Results").index
hm = (df[df["SiteId"].isin(top_sites)]
        .pivot_table(index="SiteId", columns="YearMonth", values="ValueStd", aggfunc="size", observed=True)
        .reindex(top_sites))
fig, ax = plt.subplots(figsize=(16, 9))
sns.heatmap(hm, cmap="Blues", ax=ax, cbar_kws={"label": "results in month"}, linewidths=0.5, linecolor="white")
ax.set_title("D5. Site × month sampling coverage (30 most-sampled sites; blank = no data)")
ax.set_xlabel("")
ax.set_ylabel("")
save_fig("02_D5_site_time_heatmap")
plt.show()

## 6. E — Outcome analysis: exceedances, volume vs rate (Q6)
The outcome is `ExceedanceFlag` / `HighRiskFlag` (thresholds documented in notebook 01, Section 6).
**High volume ≠ high rate.** An organization or region with many exceedances may simply sample more. Raw totals are always shown next to **normalised rates** (per 100 results, per site).

In [ ]:
thr = pd.DataFrame({
    "Unit": UNITS,
    "Exceedance when": {"Temperature": "> 28", "DO": "< 5", "pH": "< 6.5 or > 9.0", "Turbidity": "> 25",
                        "Conductivity": "> 1500 (not ocean)", "Nitrate": "> 10 (EPA MCL)"},
    "High risk when": {"Temperature": "> 32", "DO": "< 3", "pH": "< 6.0 or > 9.5", "Turbidity": "> 100",
                       "Conductivity": "> 3000 (not ocean)", "Nitrate": "> 20"}})
q6 = (df.groupby("P", observed=True)
        .agg(Results=("ValueStd", "size"), Exceedances=("ExceedanceFlag", "sum"), HighRisk=("HighRiskFlag", "sum"),
             SitesWithExceedance=("SiteId", lambda s: s[df.loc[s.index, "ExceedanceFlag"] == 1].nunique()),
             SitesMeasured=("SiteId", "nunique")))
q6["ExceedRatePct"] = 100 * q6["Exceedances"] / q6["Results"]
q6["HighRiskRatePct"] = 100 * q6["HighRisk"] / q6["Results"]
q6["PctSitesWithExceedance"] = 100 * q6["SitesWithExceedance"] / q6["SitesMeasured"]
q6 = thr.join(q6).reindex(PARAM_ORDER)
save_table(q6, "02_Q6_exceedance_by_parameter")
q6

In [ ]:
# Volume vs rate, side by side: exceedance COUNT and exceedance RATE per parameter
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4))
colors = [PARAM_COLORS[p] for p in q6.index]
a1.bar(q6.index, q6["Exceedances"], color=colors, width=0.6)
a1.set_title("E1. Exceedances: total count")
a1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x / 1e3:,.0f}k"))
a2.bar(q6.index, q6["ExceedRatePct"], color=colors, width=0.6)
a2.bar(q6.index, q6["HighRiskRatePct"], color="none", edgecolor="#0b0b0b", hatch="///", width=0.6,
       label="of which high-risk")
a2.set_title("E2. Exceedance rate (% of results); hatched = high-risk")
a2.yaxis.set_major_formatter(mtick.PercentFormatter())
a2.legend(frameon=False)
for a in (a1, a2):
    a.grid(axis="x", visible=False)
fig.tight_layout()
save_fig("02_E1_exceedance_count_vs_rate")
plt.show()

In [ ]:
# Organizations: volume vs rate (orgs with >= 1,000 results)
o = (df.groupby("OrganizationId", observed=True)
       .agg(Results=("ValueStd", "size"), Sites=("SiteId", "nunique"), Exceedances=("ExceedanceFlag", "sum")))
o = o[o["Results"] >= 1000]
o["ExceedRatePct"] = 100 * o["Exceedances"] / o["Results"]
o["ExceedPerSite"] = o["Exceedances"] / o["Sites"]
o["RankByCount"] = o["Exceedances"].rank(ascending=False).astype(int)
o["RankByRate"] = o["ExceedRatePct"].rank(ascending=False).astype(int)
save_table(o.sort_values("Exceedances", ascending=False), "02_E_org_volume_vs_rate")
display(o.sort_values("Exceedances", ascending=False).head(10))

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.scatter(o["Results"], o["ExceedRatePct"], s=np.sqrt(o["Sites"]) * 8, color=PRIMARY, alpha=0.6,
           edgecolor="white", linewidth=1)
for oid, r in o.nlargest(4, "Results").iterrows():
    ax.annotate(str(oid), (r["Results"], r["ExceedRatePct"]), fontsize=8, color="#52514e",
                xytext=(5, 5), textcoords="offset points")
for oid, r in o.nlargest(3, "ExceedRatePct").iterrows():
    ax.annotate(str(oid), (r["Results"], r["ExceedRatePct"]), fontsize=8, color="#52514e",
                xytext=(5, -10), textcoords="offset points")
ax.set_xscale("log")
ax.set_xlabel("results submitted (log scale)")
ax.set_ylabel("exceedance rate (%)")
ax.set_title("E3. Organizations: sampling volume vs exceedance rate (dot size ~ sites)")
save_fig("02_E3_org_volume_vs_rate")
plt.show()
print(f"Spearman rank correlation, volume vs rate: {o['Results'].corr(o['ExceedRatePct'], method='spearman'):.2f}")

## 7. Relationships among parameters (Q5)
Parameters are rarely measured in the same row, so they are **co-located** by **site × date**. Each pair is compared using the daily site mean (`SiteDailyAverage`). Pearson (linear) and Spearman (rank, robust to skew) correlations are both reported, together with the **number of site-days behind each pair**, so weakly supported correlations are visible.

In [ ]:
daily = (df.groupby(["SiteId", "StartDate", "P"], observed=True)["ValueStd"].mean()
           .unstack("P").reindex(columns=PARAM_ORDER))
print(f"{len(daily):,} site-days; non-missing per parameter:")
display(daily.notna().sum().to_frame("site-days"))

pearson = daily.corr(method="pearson", min_periods=200)
spearman = daily.corr(method="spearman", min_periods=200)
pair_n = daily.notna().astype(int).T @ daily.notna().astype(int)
save_table(pearson, "02_Q5_pearson")
save_table(spearman, "02_Q5_spearman")
save_table(pair_n, "02_Q5_pair_counts")

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
cmap = sns.diverging_palette(250, 25, s=80, l=50, center="light", as_cmap=True)
for ax, (name, m) in zip(axes, [("Pearson", pearson), ("Spearman", spearman)]):
    mask = np.triu(np.ones_like(m, dtype=bool))
    sns.heatmap(m, mask=mask, annot=True, fmt=".2f", cmap=cmap, vmin=-1, vmax=1, center=0, ax=ax,
                linewidths=2, linecolor="white", cbar_kws={"shrink": 0.8})
    ax.set_title(f"{name} correlation (site-day means, ≥200 shared site-days)")
fig.tight_layout()
save_fig("02_Q5_correlation_heatmaps")
plt.show()
display(pair_n)

In [ ]:
# Key pairs as density plots (hexbin: too many points for a scatter)
pairs = [("Temperature", "DO"), ("pH", "DO"), ("Turbidity", "Conductivity"), ("Nitrate", "Conductivity")]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))
for ax, (x, y) in zip(axes, pairs):
    d = daily[[x, y]].dropna()
    if x in LOG_SCALE:
        d = d[d[x] > 0]
    if y in LOG_SCALE:
        d = d[d[y] > 0]
    hb = ax.hexbin(d[x], d[y], gridsize=45, cmap="Blues", mincnt=1, bins="log",
                   xscale="log" if x in LOG_SCALE else "linear", yscale="log" if y in LOG_SCALE else "linear")
    ax.set_xlabel(f"{x} ({UNITS[x]})")
    ax.set_ylabel(f"{y} ({UNITS[y]})")
    ax.set_title(f"{x} vs {y}: ρ = {spearman.loc[x, y]:.2f}, n = {len(d):,}")
fig.tight_layout()
save_fig("02_Q5_pair_density")
plt.show()

**Expected physical check:** temperature and DO should be **negatively** correlated, because warmer water holds less oxygen. If the data shows this, it is a sign that the curation (unit conversion °F → °C, impossible-value removal) worked. The strength of other pairs depends on how many site-days actually have both parameters: always read the correlation next to its `n`.

## 8. Summary of EDA findings
The cell below **computes** the headline numbers so the summary always matches the data that was run. The interpretation follows it.

In [ ]:
def pct(x):
    return f"{x:.1f}%"


f = []
f.append(f"Analysis-ready results: {len(df):,} from {df['SiteId'].nunique():,} sites and "
         f"{df['OrganizationId'].nunique()} organizations ({df['StartDate'].min():%b %Y}–{df['StartDate'].max():%b %Y}).")
share = df["P"].value_counts(normalize=True) * 100
f.append("Parameter mix: " + ", ".join(f"{p} {share[p]:.1f}%" for p in PARAM_ORDER if p in share) + ".")
f.append(f"Top 10 organizations supply {100 * org_cov['Results'].head(10).sum() / len(df):.1f}% of results "
         f"(top: {org_cov.index[0]}, {100 * org_cov['Results'].iloc[0] / len(df):.1f}%).")
f.append("Most skewed parameters (skewness): " + ", ".join(f"{p} {num.loc[p, 'Skewness']:.1f}"
         for p in num['Skewness'].sort_values(ascending=False).index[:3]) + ".")
f.append(f"Peak sampling month: {peak.index[0]:%b %Y} ({peak.iloc[0]:,} results); "
         f"lowest: {low.index[0]:%b %Y} ({low.iloc[0]:,}).")
f.append(f"Weekend share of results: {100 * dow[['Saturday', 'Sunday']].sum() / dow.sum():.1f}%.")
f.append(f"Seasonal temperature swing (median Summer − Winter): {seasonal.loc['Temperature', 'Summer−Winter']:.1f} °C; "
         f"DO: {seasonal.loc['DO', 'Summer−Winter']:.1f} mg/L.")
f.append("Exceedance rate by parameter: " + ", ".join(f"{p} {pct(q6.loc[p, 'ExceedRatePct'])}" for p in q6.index) + ".")
f.append("Poor-quality sites (≥25% exceedances): " + ", ".join(
    f"{p} {int(r.PoorSites)}/{int(r.SitesEvaluated)}" for p, r in poor_summary.iterrows()) + ".")
f.append(f"Temperature–DO Spearman correlation: {spearman.loc['Temperature', 'DO']:.2f} "
         f"(n = {int(pair_n.loc['Temperature', 'DO']):,} site-days).")
f.append(f"Org volume vs exceedance-rate rank correlation: {o['Results'].corr(o['ExceedRatePct'], method='spearman'):.2f}, "
         "so volume and rate are different stories.")
findings = pd.DataFrame({"Finding": f}, index=range(1, len(f) + 1))
save_table(findings, "02_eda_key_findings")
with pd.option_context("display.max_colwidth", 200):
    display(findings)

### Interpretation and what it means for the next phases
1. **Data shape.** Temperature dominates the result volume, and nitrate is sparse. Models and dashboards must therefore be **per parameter**, and nitrate conclusions carry wider uncertainty.
2. **Concentration of sources.** A handful of organizations supply most results, so network-wide averages are partly *organization* averages. Dashboards should allow filtering by organization, and Phase 2 data-skew analysis should test `OrganizationId` / `LocationKey` as distribution keys.
3. **Skewed parameters** (turbidity, conductivity, nitrate) should be summarised by medians and percentiles, and log-transformed for regression in Phase 3.
4. **Temporal coverage** has strong seasonality in *sampling effort* (more field sampling in the warm season, little on weekends). Trend comparisons must use **rates/medians**, not raw counts.
5. **Seasonal signal.** Temperature rises and DO falls in summer, which matches physics. DO exceedances concentrate in the warm season, so season is a key feature for the Phase 3 decision tree.
6. **Volume ≠ rate.** Organizations and regions with the most exceedances are not always the ones with the highest exceedance *rate*. Findings must name which one they report.
7. **Relationships.** The strong negative temperature–DO relationship confirms that unit standardisation worked and makes temperature a sound predictor for the Phase 3 DO regression. Other pairs have fewer co-located site-days, so they must be read with their `n`.
8. **Gaps to close in Phase 2:** true `LocationType` and state/region come from the WQP Station file. Q9 (association rules) and Q10 (prediction/classification) are answered in Phase 3 from the warehouse analytical mart.